# **Install packages**

In [ ]:
!pip install sdv ctgan faker pandas numpy scikit-learn -q
print("✓ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 10.6 MB/s eta 0:00:00
✓ Packages installed


# **Generate normal transactions**

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

fake = Faker('en_IN')
random.seed(42)
np.random.seed(42)

# Indian UPI apps
UPI_APPS = ['PhonePe', 'GooglePay', 'Paytm', 'BHIM', 'AmazonPay']

# Indian bank VPA suffixes
VPA_SUFFIXES = ['@okicici', '@oksbi', '@okaxis', '@okhdfcbank',
                '@ybl', '@ibl', '@axl', '@paytm', '@upi']

# Merchant category codes
MCC_CODES = ['5411', '5812', '5541', '7011', '5912',
             '4111', '5999', '8099', '5311', '7372']

def generate_vpa():
    name = fake.first_name().lower().replace(' ', '')
    digits = str(random.randint(1, 9999))
    suffix = random.choice(VPA_SUFFIXES)
    return f"{name}{digits}{suffix}"

def generate_indian_ip():
    return f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(0,255)}"

def generate_gps_india():
    # India lat/lng bounds
    lat = round(random.uniform(8.0, 37.0), 6)
    lng = round(random.uniform(68.0, 97.0), 6)
    return lat, lng

# Generate 50,000 normal users
print("Generating user pool...")
users = []
for _ in range(50000):
    lat, lng = generate_gps_india()
    users.append({
        'vpa': generate_vpa(),
        'device_id': str(uuid.uuid4()),
        'ip': generate_indian_ip(),
        'lat': lat,
        'lng': lng,
        'avg_amount': random.lognormvariate(6, 1.5),  # log-normal amount dist
        'typical_hour': random.randint(8, 22)
    })

print(f"✓ {len(users)} users generated")

# Generate 950,000 normal transactions
print("Generating normal transactions...")
start_date = datetime(2024, 1, 1)
normal_txns = []

for i in range(950000):
    sender = random.choice(users)
    receiver = random.choice(users)
    while receiver['vpa'] == sender['vpa']:
        receiver = random.choice(users)

    # Amount: mostly small, occasionally large
    amount = round(abs(np.random.normal(sender['avg_amount'], sender['avg_amount']*0.3)), 2)
    amount = max(1.0, min(amount, 100000.0))

    # Timestamp: biased toward typical hour
    days_offset = random.randint(0, 364)
    hour = int(np.clip(np.random.normal(sender['typical_hour'], 2), 6, 23))
    minute = random.randint(0, 59)
    txn_time = start_date + timedelta(days=days_offset, hours=hour, minutes=minute)

    is_merchant = random.random() < 0.3
    lat, lng = generate_gps_india()

    normal_txns.append({
        'txn_id': str(uuid.uuid4()),
        'sender_vpa': sender['vpa'],
        'receiver_vpa': receiver['vpa'],
        'amount': amount,
        'timestamp': txn_time.isoformat(),
        'device_id': sender['device_id'],
        'ip_address': sender['ip'],
        'gps_lat': lat,
        'gps_lng': lng,
        'upi_app': random.choice(UPI_APPS),
        'merchant_code': random.choice(MCC_CODES) if is_merchant else None,
        'is_fraud': 0,
        'fraud_type': None
    })

    if (i+1) % 100000 == 0:
        print(f"  {i+1:,} normal transactions generated...")

normal_df = pd.DataFrame(normal_txns)
print(f"✓ {len(normal_df):,} normal transactions generated")

Generating user pool...
✓ 50000 users generated
Generating normal transactions...
  100,000 normal transactions generated...
  200,000 normal transactions generated...
  300,000 normal transactions generated...
  400,000 normal transactions generated...
  500,000 normal transactions generated...
  600,000 normal transactions generated...
  700,000 normal transactions generated...
  800,000 normal transactions generated...
  900,000 normal transactions generated...
✓ 950,000 normal transactions generated


# **Generate fraud transactions (4 types)**

In [ ]:
print("Generating fraud transactions...")
fraud_txns = []

# ── Fraud Type 1: Velocity Attack (rapid small transfers) ──────────────────
print("  Generating velocity attacks...")
for _ in range(12000):
    attacker = random.choice(users)
    burst_time = start_date + timedelta(
        days=random.randint(0, 364),
        hours=random.randint(0, 23)
    )
    num_txns = random.randint(15, 40)
    for j in range(num_txns):
        receiver = random.choice(users)
        fraud_txns.append({
            'txn_id': str(uuid.uuid4()),
            'sender_vpa': attacker['vpa'],
            'receiver_vpa': receiver['vpa'],
            'amount': round(random.uniform(1, 499), 2),
            'timestamp': (burst_time + timedelta(seconds=j*3)).isoformat(),
            'device_id': attacker['device_id'],
            'ip_address': attacker['ip'],
            'gps_lat': attacker['lat'],
            'gps_lng': attacker['lng'],
            'upi_app': random.choice(UPI_APPS),
            'merchant_code': None,
            'is_fraud': 1,
            'fraud_type': 'velocity_attack'
        })

# ── Fraud Type 2: Account Takeover (new device + new IP suddenly) ──────────
print("  Generating account takeovers...")
for _ in range(15000):
    victim = random.choice(users)
    lat, lng = generate_gps_india()
    fraud_txns.append({
        'txn_id': str(uuid.uuid4()),
        'sender_vpa': victim['vpa'],
        'receiver_vpa': generate_vpa(),
        'amount': round(random.uniform(5000, 99000), 2),
        'timestamp': (start_date + timedelta(
            days=random.randint(0, 364),
            hours=random.randint(0, 23)
        )).isoformat(),
        'device_id': str(uuid.uuid4()),   # NEW device — key signal
        'ip_address': generate_indian_ip(),  # NEW IP — key signal
        'gps_lat': lat,
        'gps_lng': lng,
        'upi_app': random.choice(UPI_APPS),
        'merchant_code': None,
        'is_fraud': 1,
        'fraud_type': 'account_takeover'
    })

# ── Fraud Type 3: Mule Chain (A→B→C→D rapid forwarding) ───────────────────
print("  Generating mule chains...")
for _ in range(8000):
    chain_len = random.randint(3, 6)
    mules = [random.choice(users) for _ in range(chain_len)]
    base_time = start_date + timedelta(days=random.randint(0, 364))
    amount = round(random.uniform(10000, 90000), 2)
    for j in range(chain_len - 1):
        fraud_txns.append({
            'txn_id': str(uuid.uuid4()),
            'sender_vpa': mules[j]['vpa'],
            'receiver_vpa': mules[j+1]['vpa'],
            'amount': round(amount * (0.9 ** j), 2),
            'timestamp': (base_time + timedelta(minutes=j*2)).isoformat(),
            'device_id': mules[j]['device_id'],
            'ip_address': mules[j]['ip'],
            'gps_lat': mules[j]['lat'],
            'gps_lng': mules[j]['lng'],
            'upi_app': random.choice(UPI_APPS),
            'merchant_code': None,
            'is_fraud': 1,
            'fraud_type': 'mule_chain'
        })

# ── Fraud Type 4: Large Unusual Transfer ───────────────────────────────────
print("  Generating large unusual transfers...")
for _ in range(15000):
    sender = random.choice(users)
    lat, lng = generate_gps_india()
    fraud_txns.append({
        'txn_id': str(uuid.uuid4()),
        'sender_vpa': sender['vpa'],
        'receiver_vpa': generate_vpa(),
        'amount': round(random.uniform(50000, 100000), 2),
        'timestamp': (start_date + timedelta(
            days=random.randint(0, 364),
            hours=random.randint(0, 5)  # unusual hour
        )).isoformat(),
        'device_id': sender['device_id'],
        'ip_address': sender['ip'],
        'gps_lat': lat,
        'gps_lng': lng,
        'upi_app': random.choice(UPI_APPS),
        'merchant_code': None,
        'is_fraud': 1,
        'fraud_type': 'large_unusual'
    })

fraud_df = pd.DataFrame(fraud_txns)
print(f"✓ {len(fraud_df):,} fraud transactions generated")
print(f"  Breakdown: {fraud_df['fraud_type'].value_counts().to_dict()}")

Generating fraud transactions...
  Generating velocity attacks...
  Generating account takeovers...
  Generating mule chains...
  Generating large unusual transfers...
✓ 386,594 fraud transactions generated
  Breakdown: {'velocity_attack': 328540, 'mule_chain': 28054, 'account_takeover': 15000, 'large_unusual': 15000}


# **Combine, engineer features, save**

In [ ]:
from sklearn.preprocessing import LabelEncoder
import json

# Combine datasets
df = pd.concat([normal_df, fraud_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total transactions: {len(df):,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

# Feature engineering
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_night'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)
df['amount_log'] = np.log1p(df['amount'])
df['is_round_amount'] = (df['amount'] % 100 == 0).astype(int)
df['has_merchant'] = df['merchant_code'].notna().astype(int)
df['upi_app_encoded'] = LabelEncoder().fit_transform(df['upi_app'])

# Velocity features (simplified — count per sender in full dataset)
sender_counts = df.groupby('sender_vpa').size().rename('sender_txn_count')
df = df.join(sender_counts, on='sender_vpa')

sender_amounts = df.groupby('sender_vpa')['amount'].mean().rename('sender_avg_amount')
df = df.join(sender_amounts, on='sender_vpa')

df['amount_vs_avg'] = df['amount'] / (df['sender_avg_amount'] + 1)

# New receiver flag (receiver seen less than 3 times)
receiver_counts = df.groupby('receiver_vpa').size().rename('receiver_count')
df = df.join(receiver_counts, on='receiver_vpa')
df['is_new_receiver'] = (df['receiver_count'] < 3).astype(int)

# Device change proxy (device_id appears with multiple VPAs)
device_vpa_counts = df.groupby('device_id')['sender_vpa'].nunique().rename('device_vpa_count')
df = df.join(device_vpa_counts, on='device_id')
df['device_shared'] = (df['device_vpa_count'] > 1).astype(int)

print("✓ Features engineered")
print(f"Feature columns: {df.shape[1]} total columns")

# Save full dataset
df.to_csv('upi_transactions_full.csv', index=False)
print("✓ Saved: upi_transactions_full.csv")

# Show class distribution
print(f"\nClass distribution:")
print(df['is_fraud'].value_counts())
print(df['fraud_type'].value_counts())

Total transactions: 1,336,594
Fraud rate: 28.92%
✓ Features engineered
Feature columns: 28 total columns
✓ Saved: upi_transactions_full.csv

Class distribution:
is_fraud
0    950000
1    386594
Name: count, dtype: int64
fraud_type
velocity_attack     328540
mule_chain           28054
large_unusual        15000
account_takeover     15000
Name: count, dtype: int64


# **Run and download**

In [ ]:
# Verify the file
import os
size_mb = os.path.getsize('upi_transactions_full.csv') / 1024 / 1024
print(f"✓ Dataset file: {size_mb:.1f} MB")
print(f"✓ Shape: {df.shape}")
print(f"\nSample fraud transaction:")
print(df[df['is_fraud']==1].iloc[0][['sender_vpa','amount','hour','fraud_type','is_night','device_shared']])

# Download the file to your computer
from google.colab import files
files.download('upi_transactions_full.csv')
print("✓ Download started")

✓ Dataset file: 340.0 MB
✓ Shape: (1336594, 28)

Sample fraud transaction:
sender_vpa       kalpit15@okaxis
amount                  60838.75
hour                           0
fraud_type            mule_chain
is_night                       1
device_shared                  0
Name: 0, dtype: object


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started
